# Progetto NLP: Filtro per Fake news
L'obiettivo è sviluppare un modello di Machine Learning che possa rilevare le fake news con precisione. Il modello sarà integrato nel plug-in Chrome e fornirà agli utenti un'indicazione in tempo reale sulla veridicità delle notizie che stanno leggendo.

Il progetto si compone nelle seguenti fasi:

1. Importazione delle librerie
2. Creazione funzioni
3. Caricamento, analisi esplorativa dei dati e preprocessing del testo <sup>1</sup>
4. Train/Test Split e Vettorizzazione
5. Addestramento dei modelli e valutazione delle performance
6. Test su nuove notizie
7. Conclusioni


N.B. Il progetto è stato esportato in formato "pickle", nonostanate la procedura e le specifiche non siano state trattate nei corsi riguardanti Python, ML e NLP.



<sup>1</sup> Utilizzando Colab ho dovuto salvare i file nel Drive, non avendo a disposizione collegamenti a repository dove poter collegare il caricamento.


1. Importazione librerie

In [ ]:
import pandas as pd
import re
import spacy
import string
import nltk
import pickle
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier

nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
english_stopwords = stopwords.words("english")
punctuation = set(string.punctuation)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2. Creazione funzioni

In [ ]:
def data_cleaner(sentence):
  """
  Ripulisce il dataset da:
  - maiuscole
  - punteggiatura
  - spazi doppi (o multipli)
  - stopwords
  Effettua la lemmatizzazione
  """
  sentence = sentence.lower()
  for c in string.punctuation:
      sentence = sentence.replace(c, " ")
  document = nlp(sentence)
  sentence = " ".join(token.lemma_ for token in document)
  sentence = " ".join(word for word in sentence.split() if word not in english_stopwords)
  sentence = re.sub(r"\d"," ", sentence)
  return sentence

In [ ]:
def bow_tfidf(dataset, tfidf_vectorizer):
  """
  Utilizza il il metodo TfidvVectorizer sul datset
  - impostazione min_dif = 0.01 (esclude parole presenti in meno dell'1% dei documenti)
  - impostazione max_df =0.90 (esclude parole presenti in più del 90% dei documenti)
  Restituisce array numpy con i vettori TF-IDF e il vettorizzatore addestrato
  """
  if tfidf_vectorizer is None:
      tfidf_vectorizer =TfidfVectorizer(min_df=0.01, max_df=0.90)
      X = tfidf_vectorizer.fit_transform(dataset)
  else:
      X = tfidf_vectorizer.transform(dataset)
  return X.toarray(), tfidf_vectorizer

In [ ]:
def analisi_news(dataset, label, n):
  """
  Analizza le parole più frequenti nei titoli delle news.
  Restituisce la lista di tuple (parola più conteggio) ordinata per frequenza decrescente
  """
  titoli_fake = " ".join(dataset[dataset["label"]==label]["title"])
  parole = titoli_fake.lower().split()
  conteggio_parole =  {}
  for parola in parole:
    if parola not in english_stopwords: # Se la parola è compresa tra le stopwords non viene conteggiata
      if parola in conteggio_parole:
        conteggio_parole[parola] +=1
      else:
        conteggio_parole[parola] = 1

  return sorted(conteggio_parole.items(), key= lambda x: x[1], reverse=True)[:n]


3. Caricamento, analisi esplorativa dei dati e processing del testo

In [ ]:
# Il progetto comprende due file divisi tra fake news e true news separati e distinti
df_fake = pd.read_csv("/content/drive/MyDrive/ColabFiles/Progetto NLP/Fake.csv")
df_true = pd.read_csv("/content/drive/MyDrive/ColabFiles/Progetto NLP/True.csv")

In [ ]:
# Verifichiamo la dimensione di entrambi i file, la presenza di colonne differenti, la presenza di valori null e i nomi delle colonne stesse
print(df_fake.shape)
print(df_true.shape)

print(df_fake.info())
print(df_true.info())

print(df_fake.columns.tolist())
print(df_true.columns.tolist())

print(df_fake.isnull().sum())
print(df_true.isnull().sum())

(23481, 4)
(21417, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB
None
['title', 'text', 'subject', 'date']
['title', 'text', 'subject', 'date']
title      0
text       0
subject    0
date       0
dtype: int64
title      0
text       0
subject    0
date       0
dtype: int64


In [ ]:
# Creo una nuova colonna in entrambi i dataset per etichettare le fake news (0) e le true news(1)

df_fake["label"] = 0
df_true["label"] = 1

In [ ]:
# Unisco i due df in uno unico
df_final = pd.concat([df_fake, df_true])

In [ ]:
# Verifico che i valori siano esattamente la somma dei due df distinti
print(df_final.shape)
print(df_final["label"].value_counts()) # Verifico che la colonna label sia correttamente distribuita

(44898, 5)
label
0    23481
1    21417
Name: count, dtype: int64


In [ ]:
# Distribuzione delle fake/true news in base al subject
print(df_final.groupby(["label", "subject"]).size())

label  subject        
0      Government News     1570
       Middle-east          778
       News                9050
       US_News              783
       left-news           4459
       politics            6841
1      politicsNews       11272
       worldnews          10145
dtype: int64


Analizzando la distribuzione sopra riportata, appare evidente che la maggior parte delle fake riguardino soprattutto news e politics rispettivamente con un 39% e un 29% del totale.
Le true news si dividono equamente (quasi 50/50) tra politicsNews e WorldNews.


In [ ]:
print("Fake News:")
analisi_news(df_final, 0, 10)

Fake News:


[('trump', 6615),
 ('[video]', 5054),
 ('(video)', 2543),
 ('obama', 1645),
 ('hillary', 1591),
 ('trump’s', 1450),
 ('watch:', 1212),
 ('president', 971),
 ('clinton', 882),
 ('new', 877)]

Senza troppe sorprese in cima alle parole più presenti tra le prime 10 nei titoli di fake news abbiamo Trump che è presente sia alla prima posizione che alla sesta (la seconda e la terza parola non sono rappresentative) Trump è seguito a lunga distanza dall'ex presidente Obama e alla candidata alla casa bianca Hillary. Comunque le fake news sono infarcite di riferimenti alla politica americana, all'attuale presidente o ad ex presidenti (o politici).

In [ ]:
print("True News:")
analisi_news(df_final, 1, 10)

True News:


[('trump', 4401),
 ('u.s.', 3872),
 ('says', 2970),
 ('house', 1383),
 ('north', 920),
 ('new', 860),
 ('white', 804),
 ('senate', 726),
 ('russia', 718),
 ('korea', 695)]

Per quanto concerne le true news l'unico politico presente nella top 10 è l'onnipresente Trump, il resto dei titoli comprende parole più varie, sempre inerenti alla politica, ma focalizzate su entità paese (korea, russia,) oppure ad organi (senato, white, house).

In [ ]:
"""
Viste le dimensioni del dataset abbiamo deciso di procedere per step, inizialmente lavorando sul 10% dello stesso,
poi sul 50% e infine sul dataset totale, il tempo per la pulizia di quest'ultimo è stata di circa 30 minuti.
"""
df_small = df_final.sample(frac=1, random_state=42)

df_small["label"].value_counts() # Verifico la distribuzione corretta dei valori

,count
label,
0,23481
1,21417


In [ ]:
# Utilizzo la funzione di data cleaner sulla colonna text, scelgo di utilizzare questa rispetto al title perchè più completa e rappresentativa.

df_small["text_clean"] = df_small["text"].apply(data_cleaner)

In [ ]:
df_small.to_csv("/content/drive/MyDrive/ColabFiles/Progetto NLP/fake_news_clean.csv") # Salvo il df pulito su un file csv

4. Train/Test Split e Vettorizzazione

In [ ]:
# Divido il dataset in features (text pulito = X) e target (labels = y)

X = df_small["text_clean"]
y = df_small["label"]

# Successivamente divido il dataset in training set e test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Vettorizzazione, utilizzo la funzione creata bow_tfidf per vettorizzare X_train e X_test
X_train_vec, tfidfv = bow_tfidf(X_train, None)
X_test_vec, _ = bow_tfidf(X_test, tfidfv)


In [ ]:
# Il vocabulary creato è di 2538 parole
len(tfidfv.vocabulary_)

2538

5. Addestramento modelli e valutazione delle performance

In [ ]:
# Avvio l'addestramento dei modelli utilizzando il ciclo per poter effettuare un confronto diretto tra le performance

models = {
    "Logistic_regression":LogisticRegression(),
    "Naive_Bayes":MultinomialNB(),
    "LinearSVC": LinearSVC(),
    "MLPClassifier": MLPClassifier(activation="logistic",
                                   solver="adam",
                                   max_iter=100,
                                   hidden_layer_sizes=(100,),
                                   tol=0.005,
                                   verbose=True)
}

for nome, model in models.items():
  model.fit(X_train_vec, y_train)
  y_pred = model.predict(X_test_vec)
  print(f"{nome}, {model.score(X_test_vec, y_test):.2f}")



Logistic_regression, 0.99
Naive_Bayes, 0.93
LinearSVC, 0.99
Iteration 1, loss = 0.51650040
Iteration 2, loss = 0.22470907
Iteration 3, loss = 0.13418007
Iteration 4, loss = 0.09733820
Iteration 5, loss = 0.07698123
Iteration 6, loss = 0.06384945
Iteration 7, loss = 0.05472000
Iteration 8, loss = 0.04788660
Iteration 9, loss = 0.04253556
Iteration 10, loss = 0.03821183
Iteration 11, loss = 0.03472309
Iteration 12, loss = 0.03166257
Iteration 13, loss = 0.02908133
Iteration 14, loss = 0.02690549
Iteration 15, loss = 0.02488347
Iteration 16, loss = 0.02311240
Iteration 17, loss = 0.02155068
Iteration 18, loss = 0.02009569
Iteration 19, loss = 0.01877095
Iteration 20, loss = 0.01766610
Training loss did not improve more than tol=0.005000 for 10 consecutive epochs. Stopping.
MLPClassifier, 0.99


In [ ]:
# Oltre ai valori di accuracy verifico anche gli altri parametri (precision, recall e f1 score) per poter confrontare meglio i modelli

for nome, model in models.items():
  print(f"\n{nome}")
  print(classification_report(y_test, model.predict(X_test_vec)))



Logistic_regression
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4710
           1       0.98      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980


Naive_Bayes
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4710
           1       0.92      0.93      0.93      4270

    accuracy                           0.93      8980
   macro avg       0.93      0.93      0.93      8980
weighted avg       0.93      0.93      0.93      8980


LinearSVC
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      4710
           1       0.99      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      

Abbiamo testato tre approcci.

10% del dataset (time cleaning 4 minuti):
* LinearSVC accuracy e f1-score = 0.98
* Ligistic Regression e MLPClassifier accuracy e f1-score = 0.97
* Naive_Bayes accuracy e f1-score 0.92

50% del dataset (time cleaning 15 minuti):
* LinearSVC accuracy e f1-score = 0.98
* Ligistic Regression e MLPClassifier accuracy e f1-score = 0.97
* Naive_Bayes accuracy 0.93 e f1-SCore 0.94

100 % del dataset (time cleaning 28 minuti):
* LinearSVC accuracy e f1-score = 0.99
* Ligistic Regression e MLPClassifier accuracy e f1-score = 0.99
* Naive_Bayes accuracy 0.93 e f1-SCore = 0.93

Nei primi due scenari il miglior modello è stato LinearSVC, mentre con il dataset completo tre modelli si sono equivalsi in tutto, LinearSVC, Logistic Regression e MLPClassifier.
Naive Bayes è stato il peggiore dei quattro in tutte le prove.
Utilizzermo LinearSVC per testare il modello su nuove news.

6. Test su nuove notizie

Scelto il modello valutiamo l'accuratezza dello stesso mettendolo alla prova con due notizie, una vera e una palesemente falsa.

In [ ]:
true = """The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday,
bringing the benchmark rate to its highest level in 15 years. Fed Chairman Jerome Powell
stated that the decision was aimed at bringing inflation back to the 2 percent target.
Markets reacted cautiously to the announcement, with stocks falling slightly."""

fake = """BREAKING: Donald Trump has revealed that the deep state is using chemtrails and
5G towers to control the minds of American citizens. Sources close to the president claim
that Bill Gates and George Soros are funding a secret government program to implant microchips
through COVID vaccines. The mainstream media is covering this up."""

In [ ]:
# Effettuiamo la pulizia di entrambe attraverso la funzione data_cleaner
fake_clean = data_cleaner(fake)
true_clean = data_cleaner(true)

In [ ]:
# Vettorizziamo le due notizie
fake_clean_vec, _ = bow_tfidf([fake_clean], tfidfv)
true_clean_vec, _ = bow_tfidf([true_clean], tfidfv)

In [ ]:
# Diamo in pasto al modello la notizia falsa vettorizzata
svc = models["LinearSVC"]
print(svc.predict(fake_clean_vec))

[0]


In [ ]:
# Stessa cosa con la notizia vera
svc = models["LinearSVC"]
print(svc.predict(true_clean_vec))

[1]


Il modello ha interpretato correttamente le due notizie, dando il label 0 alla fake news e 1 alla true news.
In precedenza abbiamo fatto un test con notizie più brevi e il modello ha avuto difficoltà ad individuare la true news. Questo ci porta ad affermare che per avere una valutazione ottimale il testo non deve essere particolarmente breve.

7. Conclusioni

Dall'analisi esplorativa del datset fornito è emerso che le fake news si concentrano principalmente su argomenti nelle categorie "News" e "Politics" e spesso fanno riferimento nei loro titoli a persone specifiche (Trump, Obama, Hillary) con un linguaggio sensazionalistico.
Le true news invece utilizzano meno riferimenti a persone particolari e più ad organizzazioni, stati e utilizzano un linguaggio più neutro e istituzionale.

Il modello più costante è risultato il LinearSVC con accuracy e f1-score che sono andati migliorando con l'aumentare dei dati utilizzati nel training, ottime prestazioni anche da Logistic Regression e MLPClassifier che hanno raggiunto i risultati di LinearSVC nel training del dataset completo.

Nella valutazione di nuove informazioni abbiamo rilevato che il modello performa peggio con testi brevi e concisi (non colto la distinzione tra fake e true in due testi di poche parole), mentre performa bene su notizie medio/lunghe.

Il dataset fornito è focalizzato principalmente sulla politica americana, quindi potrebbe non performare altrettanto bene con notizie relative ad altri ambiti.



Esportazione del modello

In [ ]:
with open ("/content/drive/MyDrive/ColabFiles/Progetto NLP/linear_svc_fakenews.pkl","wb") as f:
  pickle.dump(models["LinearSVC"], f)

with open("/content/drive/MyDrive/ColabFiles/Progetto NLP/tfidf_vectorizer.pkl", "wb") as f:
  pickle.dump(tfidfv, f)
